# 05 原作者临床版本与随访可用性

原作者网站的临床下载按钮提供Nov2023文件，整套Excel包中提供Sept2023文件；两份原始文件保留。此Notebook使用Nov2023作临床来源，影像组学特征明确保留Figshare版本标识。

这里统计的是记录和资料可用性，不是RN发生风险，也不将空白RN或存在后续MRI自动编码为阴性。

In [1]:
from pathlib import Path
import sys,json,hashlib
import pandas as pd
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from rn_public.endpoint_followup import run
sources=json.loads((ROOT/'data/metadata/endpoint_followup/molab_downloaded_sources.json').read_text())
for source in sources:
    p=ROOT/source['local_path']
    assert hashlib.sha256(p.read_bytes()).hexdigest()==source['sha256']
inventory=json.loads((ROOT/'data/metadata/endpoint_followup/molab_excel_archive_inventory.json').read_text())
sept=next(x for x in inventory['files'] if x['name']=='OpenBTAI_METS_ClinicalData_Sept2023.xlsx')
assert hashlib.sha256((ROOT/'data/raw/openbtai_molab_current'/sept['name']).read_bytes()).hexdigest()==sept['sha256']
summary=run(ROOT)
print(json.dumps(summary,indent=2))

{
  "audit_date": "2026-10-28",
  "current_clinical_source": "author-hosted file named Nov2023",
  "patients": 75,
  "lesions": 160,
  "original_rn_times": 32,
  "current_rn_times": 35,
  "current_rn_patients": 29,
  "current_unknown_rn": 125,
  "added_rn_times": 3,
  "changed_existing_rn_times": 3,
  "original_srs_records": 107,
  "current_srs_records": 115,
  "current_rn_timing_classes": {
    "unknown": 125,
    "recorded_rn_after_index_srs": 32,
    "rn_at_or_before_index_srs": 2,
    "recorded_rn_after_death": 1
  },
  "baseline_directory_candidates": 97,
  "baseline_mask_candidates": 54,
  "baseline_mask_patients": 41,
  "baseline_mask_recorded_rn_times": 18,
  "new_baseline_studies_outside_previous_archive": 4,
  "verified_non_events": 0,
  "published39_vs_current35": "unresolved_4_lesion_difference",
  "modeling_status": "blocked_no_confirmed_non_events_or_lesion_level_reference_standard",
  "models_fitted": false,
  "sept_nov_shared_field_differences": 0,
  "horizon_availabili

## 结局及治疗时间更新

对照旧Figshare文件，新版新增3条坏死时间，修改3条已有坏死时间，并补充8条首次SRS起始时间。新旧版本不混写。数据中的n/a、空格、编码和格式变化另行保留，不把所有文本差异解释成临床事实的改变。

In [2]:
changes=pd.read_csv(ROOT/'data/processed/clinical_version_changes.csv')
important=changes[changes.field.isin(['rn_time_day','first_srs_start_day'])]
print(important[['lesion_key','field','figshare_value','molab_current_value','current_source_row']].to_string(index=False))
print('Shared-field differences between September and November:',summary['sept_nov_shared_field_differences'])

lesion_key               field figshare_value molab_current_value  current_source_row
  20001_L1 first_srs_start_day            NaN               239.0                  65
  20013_L1 first_srs_start_day            NaN                55.0                  66
  20013_L2 first_srs_start_day            NaN                55.0                  67
  20013_L3 first_srs_start_day            NaN                57.0                  68
  20013_L4 first_srs_start_day            NaN                58.0                  69
  20020_L1 first_srs_start_day            NaN                59.0                  71
  40001_L1         rn_time_day            NaN               545.0                  80
  40024_L1         rn_time_day          366.0               336.0                  94
  40058_L1         rn_time_day            NaN               144.0                 107
  40076_L4         rn_time_day            NaN               531.0                 118
  40093_L3 first_srs_start_day            NaN         

## 更新后的基线特征候选

以首次记录SRS为暂定索引，MRI严格早于治疗且在其前30天内。病例必须进一步核实完整治疗史、病灶定位、影像质量与诊断依据。

In [3]:
f=pd.read_csv(ROOT/'data/processed/baseline_features_current.csv')
print('Lesions',len(f),'patients',f.patient_id.nunique(),'recorded RN times',f.rn_time_day.notna().sum())
assert len(f)==f.lesion_key.nunique()
assert f.use_for_model.sum()==0
print(f[['lesion_key','baseline_mask_date_code','first_srs_start_day','rn_time_day','rn_status']].head(8).to_string(index=False))

Lesions 54 patients 41 recorded RN times 18
lesion_key  baseline_mask_date_code  first_srs_start_day  rn_time_day           rn_status
  10005_L1                 19020128                781.0          NaN             unknown
  10029_L2                 19030605               1269.0          NaN             unknown
  10033_L2                 19000918                273.0          NaN             unknown
  10038_L1                 19001015                294.0          NaN             unknown
  10046_L1                 19000618                181.0          NaN             unknown
  20034_L1                 19000729                212.0          NaN             unknown
  20034_L2                 19000729                212.0          NaN             unknown
  40001_L2                 19000101                  8.0        168.0 recorded_event_time


## 180日与365日资料覆盖

这些是资料可用性核查窗口，不是已选定的预测期限。存在期限后的MRI只能证明有检查资料，不能证明此前未发生坏死。死亡数、MRI数及记录事件数之间可以重叠，不能相加构成竞争结局分类。

In [4]:
h=pd.read_csv(ROOT/'data/processed/horizon_availability_current.csv')
for group in ['horizon_availability_all_valid_srs','horizon_availability_baseline_mask_cohort']:
    print(group)
    print(pd.DataFrame(summary[group]).to_string(index=False))
assert h.verified_non_event.sum()==0
assert h.verified_event_free_followup_day.isna().all()

horizon_availability_all_valid_srs
 horizon_days  recorded_rn_time_within_horizon  death_recorded_by_horizon  source_scan_on_or_after_horizon_before_death  directory_scan_on_or_after_horizon_before_death  lesion_mask_on_or_after_horizon_before_death  lesion_records
          180                                6                          7                                            95                                               95                                            83             113
          365                               16                         25                                            73                                               73                                            58             113
horizon_availability_baseline_mask_cohort
 horizon_days  recorded_rn_time_within_horizon  death_recorded_by_horizon  source_scan_on_or_after_horizon_before_death  directory_scan_on_or_after_horizon_before_death  lesion_mask_on_or_after_horizon_before_death  lesion_records

## 诊断标签仍未补足

新版有35个数值型坏死时间（29名患者），125个病灶仍未知。35个时间中，32个在所记SRS之后且未晚于死亡，2个早于SRS，1个晚于死亡。时间次序合理不代表诊断依据已经逐例核实。论文报告39个病灶，与新版仍差4个，不能按患者阳性推断其余病灶阳性。

In [5]:
c=pd.read_csv(ROOT/'data/processed/clinical_current.csv')
a=pd.read_csv(ROOT/'data/processed/followup_availability_current.csv')
print(a.rn_timing_class.value_counts().to_string())
print('Unknown RN',c.rn_time_day.isna().sum(),'verified non-events',c.verified_non_event.sum())
assert c.loc[c.rn_time_day.isna(),'rn_status'].eq('unknown').all()
print('Models fitted:',summary['models_fitted'])
print('Status:',summary['modeling_status'])

rn_timing_class
unknown                        125
recorded_rn_after_index_srs     32
rn_at_or_before_index_srs        2
recorded_rn_after_death          1
Unknown RN 125 verified non-events 0
Models fitted: False
Status: blocked_no_confirmed_non_events_or_lesion_level_reference_standard
